In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab.models.galaxy_zoo import (
    GalaxyZooData,
    GalaxyZooSplit,
    GalaxyZooImages,
    GalaxyZooDataset,
)
from ugdatalab.models.galaxy_zoo.constants import N_LABELS, LABEL_COLUMNS, LABEL_DESCRIPTIVE
from ugdatalab.models.galaxy_zoo.images import _load_image
from ugdatalab.methods.cnn import baseline_rmse

import plotters

# Galaxy Image Classification — Preprocessing

This notebook handles image preprocessing (Tasks 10–13):
1. **Task 10** — Crop and resize images to reduce memory by ~30×
2. **Task 11** — Set up efficient batch loading via PyTorch DataLoader
3. **Task 12** — Split into 80% training / 20% validation and verify label distributions
4. **Task 13** — Establish baseline model (mean prediction) RMSE

In [ ]:
CSV_PATH = Path("training_classifications.csv")
IMAGE_DIR = Path("training_images")

gz = GalaxyZooData(CSV_PATH)
labels_data = np.load("galaxy_zoo_labels.npz")
labels = labels_data["labels"]
galaxy_ids = labels_data["galaxy_ids"]

print(f"Galaxies: {gz.n_galaxies}")
print(f"Labels shape: {labels.shape}")

## Task 10 — Image Downsizing

The raw SDSS images are ~424×424 pixels, most of which is empty sky. We reduce memory and computation by:

1. **Center-cropping**: removing 25% of the border on each side (keeping the central 50%, which contains the galaxy). This is justified because SDSS cutouts are centered on the target and the outer border is almost always empty sky.
2. **Resampling**: resizing the cropped image to a smaller pixel grid.

Combined, cropping by 25% per side and resizing to ~69 pixels achieves roughly a 30× reduction in total pixel count ($424^2 / 69^2 \approx 38$), while preserving enough spatial resolution for the CNN to learn morphological features.

In [ ]:
CROP_FRACTION = 0.25
TARGET_SIZE = 69

# Show before/after for a few example images
n_compare = 4
rng = np.random.default_rng(42)
compare_idx = rng.choice(gz.n_galaxies, size=n_compare, replace=False)
compare_ids = galaxy_ids[compare_idx]

images_before = [_load_image(IMAGE_DIR / f"{gid}.jpg") for gid in compare_ids]

from ugdatalab.models.galaxy_zoo.images import _crop_center, _resize
images_after = [_resize(_crop_center(img, CROP_FRACTION), TARGET_SIZE) for img in images_before]

axes = plotters.plot_image_comparison(images_before, images_after, compare_ids)
plt.show()

# Report size reduction
h_orig = images_before[0].shape[0]
reduction = (h_orig ** 2) / (TARGET_SIZE ** 2)
print(f"Original: {h_orig}x{h_orig} = {h_orig**2:,} pixels")
print(f"After crop+resize: {TARGET_SIZE}x{TARGET_SIZE} = {TARGET_SIZE**2:,} pixels")
print(f"Reduction factor: {reduction:.1f}x")

### Preprocess and save all images

We now crop and resize all images and save the result as a compressed numpy archive. This takes a few minutes but only needs to be done once — subsequent notebooks load the preprocessed arrays directly.

In [ ]:
gz_images = GalaxyZooImages(
    source=gz,
    image_dir=IMAGE_DIR,
    crop_fraction=CROP_FRACTION,
    target_size=TARGET_SIZE,
)

print(f"Preprocessed images shape: {gz_images.images.shape}")
print(f"Memory: {gz_images.images.nbytes / 1e9:.2f} GB")

np.savez_compressed(
    "galaxy_zoo_images.npz",
    images=gz_images.images,
    galaxy_ids=galaxy_ids,
)
print("Saved galaxy_zoo_images.npz")

## Task 12 — Train/Validation Split

We split the data 80/20 into training and validation sets using a random permutation with a fixed seed. After splitting, we compare the normalized label distributions of the two sets to ensure there are no systematic differences — the split should produce statistically indistinguishable distributions for all 37 labels.

In [ ]:
split = GalaxyZooSplit(gz, seed=42, train_fraction=0.8)

print(f"Training set: {len(split.train_data)} galaxies ({len(split.train_data)/gz.n_galaxies*100:.0f}%)")
print(f"Validation set: {len(split.val_data)} galaxies ({len(split.val_data)/gz.n_galaxies*100:.0f}%)")

# Compare distributions
axes = plotters.plot_split_distributions(
    split.train_labels, split.val_labels, LABEL_COLUMNS, LABEL_DESCRIPTIVE,
)
plt.show()

## Task 13 — Baseline Model

Before training any neural network, we establish a baseline: predict the training-set mean label for every image. This is the simplest possible model — it ignores the image entirely and always predicts the same label vector. Any useful CNN must outperform this baseline.

The loss function throughout this lab is the **root mean squared error** (RMSE), defined as:

$$L_{\mathrm{RMSE}} = \sqrt{\frac{1}{N_{\mathrm{galaxies}} \cdot N_{\mathrm{labels}}} \sum_i \sum_j (\ell_{\mathrm{true},ij} - \ell_{\mathrm{pred},ij})^2}$$

In [ ]:
train_rmse, val_rmse = baseline_rmse(split.train_labels, split.val_labels)
print(f"Baseline RMSE (mean prediction):")
print(f"  Training:   {train_rmse:.4f}")
print(f"  Validation: {val_rmse:.4f}")

### Save split indices and preprocessed data

In [ ]:
np.savez_compressed(
    "split_indices.npz",
    train_idx=split.train_idx,
    val_idx=split.val_idx,
    baseline_train_rmse=train_rmse,
    baseline_val_rmse=val_rmse,
)
print("Saved split_indices.npz")
print(f"  train_idx: {split.train_idx.shape}")
print(f"  val_idx: {split.val_idx.shape}")